# Tutorial · 牛津 Tutorial LLM 仿真 (v6.0)

## Persona Prompt (Oxford Tutorial Fellow + HBS Devil's Advocate)

> You are an **Oxford tutorial fellow** in **Agent框架对比 (LangGraph/CrewAI/AutoGen/MetaGPT, orchestration patterns)**.
> **Never give direct answers.** Use **Socratic questioning** to make the student reason.
> Act as an **HBS devil's advocate**: challenge assumptions, demand evidence, reject hand-waving.
> **Reject vague claims** (e.g. "LangGraph is more flexible" without specifying which axis of flexibility).
> **End each turn with a probing question.**

### 你扮演的导师画像
- 学院: Oxford-style one-on-one tutorial (师生比 1:1, 每周 1 次)
- 风格: HBS case-method devil's advocate (不接受"我觉得", 要数据/反例/因果链)
- 立场: 中立 (不预设 LangGraph 比 CrewAI 好, 逼学生自己推演)
- 工具: 苏格拉底 5 问 (为什么/反例/若前提变/凭什么/如何)
- 限频: 每单元 1 次/天 (防依赖, 见 cell6)

### 学习科学依据
- **Oxford tutorial** (1:1, 高反馈密度) -> Hattie effect size 高
- **HBS devil's advocate** -> 强制学生面对反例, 避免确认偏误
- **Hattie 4 级反馈** (Task/Process/Self-Reg/Feed-Forward) -> 避免空洞 Self 级表扬
- **Pre-tutorial retrieval** (cell2) -> 提取练习优于重读

> 本 notebook 是**静态仿真**: Socratic loop 用 if/else 模拟导师追问, 不真调 openai/anthropic API。学生答案输入后, 导师按答案分支给出追问/反馈。


## Cell 2 · Pre-Tutorial Task (强制 Retrieval, 提取练习)

> 牛津 tutorial 前必须交一份 short essay/解题/方案。本单元 pre-tutorial task:

### 任务: 提交一段 200 字方案 (retrieval practice)

**题目**: 同一个营销任务(透肌精华竞品分析+策略生成), 你会在 LangGraph ReAct / LangGraph Plan-Execute / CrewAI / AutoGen 中选哪个? 用天道推演的因果链视角说明理由, 标注不可逆点。

**要求** (任一不满足, tutorial 拒绝开始):
1. 必须引用 >=2 个框架的 API (如 `StateGraph` / `create_react_agent` / `Agent(role,goal,backstory)` / `GroupChat(max_round=...)`)
2. 必须标注 >=1 个不可逆点 (如 plan_node 一次性规划 / max_round 耗尽 / 角色定义错误)
3. 必须给出 >=2 个差异化策略选项 (不可只给单一结论)
4. 必须承认 >=1 个认知盲点 (如 StubLLM 无法真实模拟 LLM 推理质量)

**提交方式**: 在下方 cell3 把方案字符串赋给 `student_essay`, 运行 cell3 进入 Socratic loop。

> 为什么强制 retrieval? 因为重读 notes.md 的迁移效果远低于主动提取。Pre-tutorial task 强制你从长期记忆提取框架语义与因果链, 即便写得不对, 提取动作本身已强化记忆 (testing effect)。


In [ ]:
# Cell 3 · Multi-turn Socratic Loop (静态 if/else 仿真, >=4 轮, >=5 苏格拉底问)
# 不调 openai/anthropic API。学生输入 essay, 导师按答案分支追问。

student_essay = """
我会选 LangGraph Plan-Execute。因为透肌精华竞品分析是结构化任务,
信息充分时一次性规划4步(search->analyze->write->END)效率高。
plan_node是不可逆点, 前提错误会传播到所有execute步骤。
CrewAI的控制流由Task的context依赖决定, 不如LangGraph精确。
但我承认StubLLM无法真实模拟LLM推理质量。
"""

# 苏格拉底问计数器 (确保 >=5 个: 为什么/反例/若前提变/凭什么/如何)
socratic_questions_asked = []
def ask(tag, question):
    socratic_questions_asked.append((tag, question))
    print(f"[TUTOR · {tag}] {question}")

# Round 1: 为什么 (Why)
print("=" * 70)
print("ROUND 1")
print("=" * 70)
print(f"[STUDENT ESSAY] {student_essay.strip()}")
print()
ask("为什么", "你说'信息充分时一次性规划效率高'——凭什么判断透肌精华竞品分析'信息充分'? "
            "如果是新品类(如2026年某成分刚爆火), 竞品数据稀缺, Plan-Execute的plan_node还成立吗?")

# 学生分支1: 是否承认信息充分性是假设
r1_answer = "信息充分是我的假设, 新品类可能不成立"  # 模拟学生回答
print(f"[STUDENT R1] {r1_answer}")
print()

# Round 2: 反例 (Counterexample)
print("=" * 70)
print("ROUND 2")
print("=" * 70)
ask("反例", "好。那给一个反例: 如果改用 ReAct 模式(边推理边调工具), 在新品类场景下, "
           "Agent 会在第几步发现'竞品数据稀缺'? 这个发现能回传修正 plan 吗? "
           "ReAct 的 Thought->Action->Obs 循环如何处理这个动态性?")

r2_answer = "ReAct 在 search_product_info 步就能发现数据稀缺, 但Plan-Execute的plan_node已固定4步"
print(f"[STUDENT R2] {r2_answer}")
print()

# Round 3: 若前提变 (What if premise changes)
print("=" * 70)
print("ROUND 3")
print("=" * 70)
ask("若前提变", "现在前提变了: 假设透肌精华是成熟品类(数据充分), 但营销策略需要多视角辩论(品牌/价格/渠道三方)。"
              "此时 Plan-Execute 还是最优吗? AutoGen 的 GroupChat(max_round=6) 在多视角辩论上是不是更合适? "
              "为什么? 用因果链说明。")

r3_answer = "成熟品类+多视角辩论, AutoGen更优, 因果链在对话涌现, Plan-Execute的plan_node反而限制了视角"
print(f"[STUDENT R3] {r3_answer}")
print()

# Round 4: 凭什么 (On what grounds) + 如何 (How)
print("=" * 70)
print("ROUND 4")
print("=" * 70)
ask("凭什么", "你说 AutoGen 更优——凭什么? AutoGen 的 max_round=6 耗尽仍未共识怎么办? "
            "这是不是新的不可逆点? 你essay里只标注了plan_node一个不可逆点, "
            "AutoGen的max_round耗尽是不是第二个不可逆点? 凭什么忽略它?")
ask("如何", "如何在你essay的'>=2个差异化策略选项'里同时容纳 Plan-Execute(结构化) "
           "和 AutoGen(多视角辩论) 两个方案, 并给出切换条件? "
           "这个切换条件本身是不是一个高杠杆点?")

r4_answer = "max_round耗尽是第二不可逆点, 我essay遗漏了。切换条件=任务结构化程度×多视角需求"
print(f"[STUDENT R4] {r4_answer}")
print()

# Round 5: 收尾追问 (确保 >=5 苏格拉底问)
print("=" * 70)
print("ROUND 5 (收尾)")
print("=" * 70)
ask("为什么", "最后一个: 你说切换条件=任务结构化程度×多视角需求——这个二元条件够吗? "
            "CrewAI的角色分工在什么场景下比Plan-Execute和AutoGen都优? "
            "为什么你的essay完全没提CrewAI的适用边界?")

r5_answer = "CrewAI适合角色明确可分工, 我essay遗漏了CrewAI的因果链分析"
print(f"[STUDENT R5] {r5_answer}")
print()

# 苏格拉底问统计
print("=" * 70)
print(f"苏格拉底问总数: {len(socratic_questions_asked)} (要求 >=5)")
for tag, q in socratic_questions_asked:
    print(f"  - [{tag}] {q[:50]}...")
assert len(socratic_questions_asked) >= 5, "苏格拉底问不足5个!"
print("PASS: >=5 苏格拉底问已触发")


In [ ]:
# Cell 4 · student_model.json 读写 (记录掌握度/盲点)
# 不调 API。纯本地 JSON 读写, 持久化学生模型, 供下次 tutorial 优先追问盲点。

import json
from pathlib import Path

STUDENT_MODEL_PATH = Path("./student_model.json")

# 初始化 student_model (若不存在)
def load_student_model():
    if STUDENT_MODEL_PATH.exists():
        return json.loads(STUDENT_MODEL_PATH.read_text(encoding="utf-8"))
    return {
        "unit": "U-E1-D2",
        "subskills": {
            "S-A-graph-orchestration": {"mastery": 0.0, "attempts": 0, "weak": False},
            "S-B-role-dialogue-api": {"mastery": 0.0, "attempts": 0, "weak": False},
            "S-C-causal-selection": {"mastery": 0.0, "attempts": 0, "weak": False},
        },
        "blind_spots": [],
        "last_tutorial": None,
    }

# 从 Socratic loop 推断盲点 (静态规则)
def infer_blind_spots(essay, socratic_rounds):
    blinds = []
    # 规则1: essay 只提 Plan-Execute, 没提 CrewAI 适用边界
    if "CrewAI" not in essay or "CrewAI" in essay and "适用边界" not in essay:
        blinds.append({
            "id": "BS1",
            "desc": "未分析 CrewAI 角色化分工的适用边界, 选型论证遗漏一框架",
            "subskill": "S-C-causal-selection",
            "remediation": "回看 practice.md Drill C1 worked: 四框架时间线推演模板",
        })
    # 规则2: 只标注 1 个不可逆点 (plan_node), 漏 max_round 耗尽
    if "max_round" not in essay:
        blinds.append({
            "id": "BS2",
            "desc": "不可逆点只标 plan_node, 漏 AutoGen max_round 耗尽",
            "subskill": "S-C-causal-selection",
            "remediation": "回看 notes.md §天道推演: AutoGen 路径的不可逆点",
        })
    # 规则3: 没提 ReAct 与 Plan-Execute 的动态性对比
    if "ReAct" not in essay or "动态" not in essay:
        blinds.append({
            "id": "BS3",
            "desc": "未对比 ReAct 的动态发现 vs Plan-Execute 的固定 plan",
            "subskill": "S-A-graph-orchestration",
            "remediation": "回看 starter.ipynb TODO2/TODO3: 两版真实代码对比",
        })
    return blinds

# 更新 student_model
student_essay = student_essay  # 来自 cell3
blinds = infer_blind_spots(student_essay, socratic_questions_asked)

model = load_student_model()
model["blind_spots"] = blinds
model["last_tutorial"] = "2026-07-26"
# 弱项循环触发: 连续2次失败 -> weak=True
for bs in blinds:
    sk = bs["subskill"]
    if sk in model["subskills"]:
        model["subskills"][sk]["attempts"] += 1
        if model["subskills"][sk]["attempts"] >= 2 and model["subskills"][sk]["mastery"] < 0.8:
            model["subskills"][sk]["weak"] = True

STUDENT_MODEL_PATH.write_text(json.dumps(model, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"[student_model.json] 已写入: {STUDENT_MODEL_PATH}")
print(f"盲点数: {len(blinds)}")
for bs in blinds:
    print(f"  - {bs['id']}: {bs['desc']}")
    print(f"    -> remediation: {bs['remediation']}")
print(f"弱项循环触发: {[k for k,v in model['subskills'].items() if v['weak']]}")


In [ ]:
# Cell 5 · Hattie 4 级 Formative Feedback (Task/Process/Self-Reg/Feed-Forward)
# 避免 Self 级空洞表扬 (Hattie: Self 级表扬 effect size 低, 甚至负效)
# 4 个标记必含: [TASK] / [PROCESS] / [SELF-REG] / [FEED-FORWARD]

hattie_feedback = """
==================================================================
Hattie 4 级 Formative Feedback (U-E1-D2 Agent框架对比)
==================================================================

[TASK] 任务级反馈 (针对本次 essay 的具体内容):
  - 你的 essay 正确标注了 plan_node 是 Plan-Execute 的不可逆点,
    这是准确的因果链识别。
  - 但 essay 遗漏了 AutoGen max_round 耗尽这一第二不可逆点,
    导致选型论证在"多视角辩论"场景下论证不完整。
  - essay 完全未分析 CrewAI 的适用边界, 选型只在 3 框架(漏 CrewAI)中展开,
    不满足">=2 差异化策略"的完备性。

[PROCESS] 过程级反馈 (针对你推演因果链的方法):
  - 你的推演方法: 先选框架再找不可逆点。这是"确认偏误"路径。
  - 建议方法: 先列所有不可逆点候选(plan_node/max_round/角色定义错误/ReAct循环不终止),
    再对每个不可逆点评估各框架的脆弱性, 最后选脆弱性最低的框架。
  - 这个"先发散后收敛"的推演方法来自天道推演的沙盘展开阶段。

[SELF-REG] 自我调节级反馈 (针对你的元认知):
  - 你在 R4 才承认 max_round 是第二不可逆点——这意味着你在 essay 阶段
    就隐约意识到但未写出。Self-reg 的关键是: 写 essay 时就问自己
    "我标注的不可逆点够全吗? 有没有第二个?"。
  - 建议每次 essay 提交前自问: ">=2 不可逆点? >=2 策略? >=1 盲点?"
    三问全 yes 才提交。这叫 pre-submission self-check。

[FEED-FORWARD] 前馈级反馈 (针对下一步行动):
  - 下一步: 重做 practice.md Drill C1 independent, 用新变体
    (把透肌精华换成另一个竞品, 如"敏感肌修护霜竞品分析"),
    重新推演 4 框架时间线, 标注 >=2 不可逆点。
  - 进 Day 3 (多 Agent 系统设计) 前, 必须先 mastery S-C (因果链选型),
    否则 Day 3 的"通信协议/共识机制/冲突解决"会因选型能力不足而卡壳。
  - 推荐复习: schedule.json C4 (AutoGen max_round) + C7 (天道推演因果链选型)
    间隔重复 due=[1,3,8,21,60,180] 天。

(注: 本反馈刻意避免 Self 级表扬如"你做得很好"。Hattie 元分析显示
 Self 级表扬 effect size 低, 真正有效的是 Task/Process/Self-Reg/Feed-Forward。)
==================================================================
"""
print(hattie_feedback)

# 验证 4 标记存在
required_tags = ["[TASK]", "[PROCESS]", "[SELF-REG]", "[FEED-FORWARD]"]
for tag in required_tags:
    assert tag in hattie_feedback, f"缺少 Hattie 标记: {tag}"
print("PASS: 4 个 Hattie 标记全部存在")


## Cell 6 · 限频 + Exit Artifact

### 限频 (防依赖)
- **每单元 1 次/天**: 本 tutorial 每天最多 1 次。原因: 防"依赖导师追问才思考"。
- **强制间隔**: 两次 tutorial 间隔 >=24h。期间用 schedule.json 间隔重复 (FSRS-6) 巩固。
- **退出条件**: 当 student_model.json 的 3 个 subskill 全部 mastery>=0.8 且 blind_spots=[], 本单元 tutorial 关闭, 进入 Day 3。

### Exit Artifact (每次 tutorial 结束必交)
完成本次 tutorial 后, 在下方提交 exit artifact (写进 student_model.json):

**必含 3 项**:
1. **2-3 盲点** (从 cell4 的 infer_blind_spots 取, 或自行补充):
   - 例: BS1 未分析 CrewAI 适用边界 / BS2 漏 max_round 不可逆点 / BS3 未对比 ReAct 动态性
2. **推荐复习单元** (跨单元回链):
   - 本单元盲点 -> 复习: elective-e1-agentic-ai/day-1-agent-theory (ReAct/Plan-Execute 范式基础)
   - AutoGen 盲点 -> 预习: elective-e1-agentic-ai/day-3-multi-agent-systems (多 Agent 通信协议)
3. **下次 tutorial 的 pre-task** (>=1 个 retrieval 题):
   - 例: "默写 CrewAI 的 Agent/Task/Crew 三件套 API 签名, 不查 notes.md"

### 退出本次 tutorial 的检查清单
- [ ] cell3 Socratic loop 跑完 >=4 轮, >=5 苏格拉底问触发
- [ ] cell4 student_model.json 已写入, 盲点数 >=1
- [ ] cell5 Hattie 4 标记全部存在
- [ ] exit artifact 3 项已提交

> 完成 checklist 后, 本日 tutorial 额度用尽。下次 tutorial 须隔 >=24h, 期间用 schedule.json 间隔重复。

---

*本 notebook 落实 Oxford tutorial + HBS devil's advocate + Hattie 4 级反馈。静态 if/else 仿真, 不调真实 LLM API。学习科学依据见 practice.md / alignment.md / schedule.json。*
